# imports

In [1]:
# Imports
import numpy as np
from numpy import nan
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats
from scipy.spatial.distance import cdist
from tqdm.auto import tqdm
import seaborn as sns
import importlib
from scipy.stats import bernoulli
import warnings
from plotter import *
from scipy.special import softmax
from scipy.spatial.distance import cdist
import gymnasium as gym
from gymnasium.envs.registration import register, registry, make, spec
import pickle
import copy
from itertools import product
import json
from functools import partial
from scipy.optimize import Bounds, minimize, differential_evolution
import multiprocess as mp
from pybads import BADS
import pymer4.models as pm

import IPython

import multiprocess as mp
import pingouin as pg
from scipy.special import expit

from agents import Farmer, BAMCP, CE
from utils import *
from runners import run_grid, run_bandit, run_emp, run_emp_enum
from MCTS import MonteCarloTreeSearch_AFC, MonteCarloTreeSearch_Bandit, MonteCarloTreeSearch_Emp


warnings.filterwarnings('ignore')


%load_ext autoreload
%autoreload 2

In [68]:
room_1 = [
    [1,0,0],
    [0,1,0]
    ]
room_2 = [
    [1,0,0],
    [0, 0.5,0.5],
    ]
ells = [0.5, 1.0, 2]
for ell in ells:
    for r, room in enumerate([room_1, room_2], start=1):
        print(f"Empowerment for ell={ell} in room {r}: {env.empowerment(room, ell)}")
    print('')

Empowerment for ell=0.5 in room 1: 2.0
Empowerment for ell=0.5 in room 2: 2.414213562373095

Empowerment for ell=1.0 in room 1: 2.0
Empowerment for ell=1.0 in room 2: 2.0

Empowerment for ell=2 in room 1: 2.0
Empowerment for ell=2 in room 2: 1.5



In [48]:
## make emp env
n_arms = 3
n_outcomes = 5
n_trials = 3
alpha = 1
ell = 0.9
env = make_emp_env(n_arms, n_outcomes, n_trials, alpha, ell, seed = 1)

## make agent
agent = BAMCP(
    mcts_class=MonteCarloTreeSearch_Emp, run_fn=run_emp,
    n_samples=100000,
    exploration_constant=3,
    discount_factor=1,
    horizon=1,
    temp=1,
    lapse=0,
)


run_emp_enum(agent, env)

initial emp: 1.174618943088019
  trial   1/3  Q=[1.4532 1.4532 1.4532]  pulled arm 0, outcome 1, empowerment reward 1.3117
  trial   2/3  Q=[1.4479 1.4558 1.4558]  pulled arm 1, outcome 4, empowerment reward 1.4489
  trial   3/3  Q=[1.4927 1.4927 1.4458]  pulled arm 1, outcome 1, empowerment reward 1.4007


{'Q': array([[1.45316539, 1.45316539, 1.45316539],
        [1.44787919, 1.45580849, 1.45580849],
        [1.49266005, 1.49266005, 1.44579923]]),
 'p_choice': array([[0.33333333, 0.33333333, 0.33333333],
        [0.3315736 , 0.3342132 , 0.3342132 ],
        [0.3384988 , 0.3384988 , 0.3230024 ]]),
 'actions': array([0, 1, 1]),
 'outcomes': array([1, 4, 1]),
 'rewards': array([1.31173621, 1.44885348, 1.40065859]),
 'cumulative_reward': array([1.31173621, 2.76058969, 4.16124828]),
 'true_p_matrix': array([[2.31335540e-01, 5.46232886e-01, 4.90366701e-05, 1.54341817e-01,
         6.80407205e-02],
        [4.82877216e-02, 1.02729236e-01, 2.11313379e-01, 2.51921721e-01,
         3.85747943e-01],
        [1.33813905e-01, 2.84670660e-01, 5.63303668e-02, 5.18345951e-01,
         6.83911773e-03]]),
 'posterior_p_matrix': array([[0.16666667, 0.33333333, 0.16666667, 0.16666667, 0.16666667],
        [0.14285714, 0.28571429, 0.14285714, 0.14285714, 0.28571429],
        [0.2       , 0.2       , 0.2    